# Makemore MLP and BatchNorm in the browser

A compact, editable version of the next Zero-to-Hero compatibility slice. Increase `steps` when experimenting.

In [ ]:
%pip install -q torchlite
import math
import torch
import torch.nn.functional as F
print('torchlite', torch.__version__)

In [ ]:
words = ['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']
chars = sorted(set(''.join(words)))
stoi = {char: index + 1 for index, char in enumerate(chars)}
stoi['.'] = 0
block_size = 3
X, Y = [], []
for word in words:
    context = [0] * block_size
    for char in word + '.':
        index = stoi[char]
        X.append(context)
        Y.append(index)
        context = context[1:] + [index]
X, Y = torch.tensor(X), torch.tensor(Y)
vocab_size = len(stoi)
print(X.shape, Y.shape, vocab_size)

In [ ]:
g = torch.Generator().manual_seed(2147483647)
n_embd, n_hidden = 8, 48
C = torch.randn((vocab_size, n_embd), generator=g)
W1 = torch.randn((block_size * n_embd, n_hidden), generator=g)
W1.data *= (5 / 3) / math.sqrt(block_size * n_embd)
b1 = torch.zeros(n_hidden)
W2 = torch.randn((n_hidden, vocab_size), generator=g)
W2.data *= 0.01
b2 = torch.zeros(vocab_size)
parameters = [C, W1, b1, W2, b2]
for parameter in parameters:
    parameter.requires_grad = True

steps = 120
losses = []
for step in range(steps):
    emb = C[X]
    hidden = torch.tanh(emb.view(-1, block_size * n_embd) @ W1 + b1)
    logits = hidden @ W2 + b2
    loss = F.cross_entropy(logits, Y)
    losses.append(loss.item())
    for parameter in parameters:
        parameter.grad = None
    loss.backward()
    for parameter in parameters:
        parameter.data += -0.2 * parameter.grad

print(f'mlp loss: {losses[0]:.4f} -> {losses[-1]:.4f}')
assert losses[-1] < losses[0] * 0.8

In [ ]:
probe = torch.randn((16, 8), generator=g, requires_grad=True)
gain = torch.ones((1, 8), requires_grad=True)
bias = torch.zeros((1, 8), requires_grad=True)
normalized = (probe - probe.mean(0, keepdim=True)) / torch.sqrt(
    probe.var(0, keepdim=True, unbiased=True) + 1e-5
)
bn_output = gain * normalized + bias
bn_loss = (bn_output ** 2).mean()
bn_output.retain_grad()
bn_loss.backward()
assert probe.grad is not None and gain.grad is not None and bias.grad is not None
print(f'batchnorm gradients: ok; output std={bn_output.std().item():.4f}')